# **Predicting Online Shoppers' Purchasing Intention Using Random Forest and AdaBoost Classifiers**

## **1. Introduction**

In [ ]:
import pandas as pd
import numpy as np

from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_curve, roc_auc_score

from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [ ]:
# create a dataframe

df = pd.read_csv('../data/online_shoppers_intention_data_set.csv', sep=";")

## **2. Data Understanding**

In [ ]:
# view first 5 rows of dataset

df.head()

In [ ]:
# understand the types of variables in the dataset.

df.info()

In [ ]:
# view the descriptive statistics of the variables in the data set.

df.describe()

In [ ]:
# check for null values

df.isnull().sum()

In [ ]:
# check total count for revenue 'True' vs 'False

df['Revenue'].value_counts()

## **3. Data Preperation**

In [ ]:
# check for and handle duplicate rows

duplicates = df.duplicated().sum()

print(f"Duplicate rows found: {duplicates}")

### **3.1 Encode the Target Variable**

In [ ]:
# convert the the Revenue categories to numerical values.

df['Revenue'] = df['Revenue'].map({True: 1, False: 0})

In [ ]:
# check the class distribution

Counter(df['Revenue'])

### **3.2 Encode the categorical features**

In [ ]:
# Convert specific columns to one-hot encoding
# drop_first = True - avoid multicollinearity

df = pd.get_dummies(df, columns=['Month', 'VisitorType'],
                    drop_first=True)

### **3.3 Encode the Boolean Variable**

In [ ]:
# convert the the Weekend categories to numerical values.

df['Weekend'] = df['Weekend'].map({True: 1, False: 0})

In [ ]:
# confirm the coding was sucessful

df.head()

### **3.4 Scale the numerical values**

In [ ]:
# transform each numerical variable so it has mean = 0 and std = 1,
# so large numbers do not dominate the model

scaler = StandardScaler()
columns_to_scale = ['Administrative', 'Administrative_Duration',
    'Informational', 'Informational_Duration',
    'ProductRelated', 'ProductRelated_Duration',
    'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay'
]
df[columns_to_scale] = scaler.fit_transform(df[columns_to_scale])

In [ ]:
# confirm the scaling was successful

df.head()

### **3.5 Train-Test Split**

In [ ]:
# separate into input (X) and output (y)

y = df['Revenue']       # output
X = df.drop('Revenue', axis=1)  # inputs

In [ ]:
# split the data

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# validate the data was split correctly

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

## **4. Random Forest**

### **4.1 Baseline Model**

In [ ]:
# create the baseline model

baseline_rf = Pipeline([
    ('balancing', SMOTE(random_state=42)),
    ('classification', RandomForestClassifier(criterion='entropy', max_features='sqrt', random_state=42))
])

# train the model

baseline_rf.fit(X_train, y_train)

# make predictions on the test set

baseline_rf_predictions = baseline_rf.predict(X_test)

In [ ]:
# print baseline performance

print("Baseline Model:")
print(classification_report(y_test, baseline_rf_predictions))

### **4.2 Hyperparameter Tuning - Round 1 (n_estimators)**

In [ ]:
# Define the model pipeline

model_rf = Pipeline([
    ('balancing', SMOTE(random_state=42)),
    ('classification', RandomForestClassifier(criterion='entropy', max_features='sqrt', random_state=42))
])

In [ ]:
# Define parameter grid

param_grid_rf = {
    'classification__n_estimators': [50, 100, 200, 300]
}

In [ ]:
# Run GridSearchCV using scoring='recall'

grid_search_rf = GridSearchCV(
    estimator=model_rf,
    param_grid=param_grid_rf,
    scoring='recall',
    cv=5
)

grid_search_rf.fit(X_train, y_train)

In [ ]:
# Print best parameters and score

print(grid_search_rf.best_params_)
print(grid_search_rf.best_score_)

In [ ]:
model_rf_predictions = grid_search_rf.predict(X_test)

print("Hyperparameter Tuning Model - Round 1:")
print(classification_report(y_test, model_rf_predictions))

### **4.3 Feature Importance**

In [ ]:
# extract feature importances from the best model.

featimp = pd.Series(
    grid_search_rf.best_estimator_.named_steps['classification'].feature_importances_,
    index=list(X)
).sort_values(ascending=False)

print(featimp)

### **4.4 Hyperparameter Tuning — Round 2 (Top Features)**

In [ ]:
# select the top features based on importance scores

top_features = featimp.head(8).index.tolist()

X_train_ = X_train[top_features]
X_test_ = X_test[top_features]

In [ ]:
# retrain with reduced features and a higher range of n_estimators

model_rf_2 = Pipeline([
    ('balancing', SMOTE(random_state=42)),
    ('classification', RandomForestClassifier(criterion='entropy', max_features='sqrt', random_state=42))
])

param_grid_rf_2 = {
    'classification__n_estimators': [200, 250, 300, 350, 400]
}

grid_search_rf_2 = GridSearchCV(
    estimator=model_rf_2,
    param_grid=param_grid_rf_2,
    scoring='recall',
    cv=5
)

grid_search_rf_2.fit(X_train_, y_train)

In [ ]:
# print the best parameters and score for round 2

print("Round 2 - Best Parameters:", grid_search_rf_2.best_params_)
print("Round 2 - Best CV Score:", grid_search_rf_2.best_score_)

In [ ]:
# evaluate on the test set with reduced features

tuned_rf_predictions_2 = grid_search_rf_2.predict(X_test_)

print("Hyperparameter Tuning Model - Round 2:")
print(classification_report(y_test, tuned_rf_predictions_2))

## **5. AdaBoost**

### **5.1 Baseline Model**

In [ ]:
# create the baseline model

baseline_ab = Pipeline([
    ('balancing', SMOTE(random_state=42)),
    ('classification', AdaBoostClassifier(random_state=42))
])

# train the model

baseline_ab.fit(X_train, y_train)

# make predictions on the test set

baseline_ab_predictions = baseline_ab.predict(X_test)

In [ ]:
# print baseline performance

print("Baseline Model:")
print(classification_report(y_test, baseline_ab_predictions))

### **5.2 Hyperparameter Tuning - Round 1 (n_estimators)**

In [ ]:
# Define the model pipeline

model_ab = Pipeline([
    ('balancing', SMOTE(random_state=42)),
    ('classification', AdaBoostClassifier(random_state=42))
])

In [ ]:
# Define parameter grid

param_grid_ab = {
    'classification__n_estimators': [50, 100, 200, 300],
     'classification__learning_rate': [0.01, 0.1, 0.5, 1.0]
}

In [ ]:
# Run GridSearchCV using scoring='recall'

grid_search_ab = GridSearchCV(
    estimator=model_ab,
    param_grid=param_grid_ab,
    scoring='recall',
    cv=5
)

grid_search_ab.fit(X_train, y_train)

In [ ]:
# Print best parameters and score

print(grid_search_ab.best_params_)
print(grid_search_ab.best_score_)

In [ ]:
model_ab_predictions = grid_search_ab.predict(X_test)

print("Hyperparameter Tuning Model - Round 1:")
print(classification_report(y_test, model_ab_predictions))

### **5.3 Features Importance**

In [ ]:
# extract feature importances from the best model.

featimp_ab = pd.Series(
    grid_search_ab.best_estimator_.named_steps['classification'].feature_importances_,
    index=list(X)
).sort_values(ascending=False)

print(featimp_ab)

### **5.4 Hyperparameter Tuning - Round 2 (Top Features)**

In [ ]:
# select the top features based on importance scores

top_features_ab = featimp_ab.head(8).index.tolist()

X_train_ = X_train[top_features_ab]
X_test_ = X_test[top_features_ab]

In [ ]:
# retrain with reduced features and a higher range of n_estimators

model_ab_2 = Pipeline([
    ('balancing', SMOTE(random_state=42)),
    ('classification',AdaBoostClassifier(random_state=42))
])

param_grid_ab_2 = {
    'classification__n_estimators': [200, 250, 300, 350, 400],
     'classification__learning_rate': [0.01, 0.1, 0.5, 1.0]
}

grid_search_ab_2 = GridSearchCV(
    estimator=model_ab_2,
    param_grid=param_grid_ab_2,
    scoring='recall',
    cv=5
)

grid_search_ab_2.fit(X_train_, y_train)

In [ ]:
# print the best parameters and score for round 2

print("Round 2 - Best Parameters:", grid_search_ab_2.best_params_)
print("Round 2 - Best CV Score:", grid_search_ab_2.best_score_)

In [ ]:
# evaluate on the test set with reduced features

tuned_ab_predictions_2 = grid_search_ab_2.predict(X_test_)

print("Hyperparameter Tuning Model - Round 2:")
print(classification_report(y_test, tuned_ab_predictions_2))

## **6. Model Evaluation and Comparison**

### **6.1 Confusion Matrices**

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ConfusionMatrixDisplay.from_predictions(y_test, model_rf_predictions, ax=axes[0])
axes[0].set_title("RandomForest: Hyperparameter - Round 1")

ConfusionMatrixDisplay.from_predictions(y_test, tuned_ab_predictions_2, ax=axes[1])
axes[1].set_title("AdaBoost: Hyperparameter - Round 2")

plt.tight_layout()
plt.show()

In [ ]:
# calculate ROC curve and AUC for Random Forest (Hyperparameter Tuning - Round 1)

y_proba_rf = grid_search_rf.predict_proba(X_test)[:, 1]
fpr_rf, tpr_rf, thresholds_rf = roc_curve(y_test, y_proba_rf)
auc_score_rf = roc_auc_score(y_test, y_proba_rf)

plt.figure(figsize=(8, 6))
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {auc_score_rf:.2f})')

# calculate ROC curve and AUC for AdaBoost (Hyperparameter Tuning - Round 2)

y_proba_ab = grid_search_ab_2.predict_proba(X_test_)[:, 1]
fpr_ab, tpr_ab, thresholds_ab = roc_curve(y_test, y_proba_ab)
auc_score_ab = roc_auc_score(y_test, y_proba_ab)

plt.plot(fpr_ab, tpr_ab, label=f'AdaBoost (AUC = {auc_score_ab:.2f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.show()

In [ ]:
# final results summary

results = pd.DataFrame({
    'Model': ['RF Baseline', 'RF Tuned (Round 1)', 'RF Tuned (Round 2)',
              'AB Baseline', 'AB Tuned (Round 1)', 'AB Tuned (Round 2)'],
    'Precision (Class 1)': [0.63, 0.63, 0.59, 0.60, 0.60, 0.58],
    'Recall (Class 1)': [0.74, 0.75, 0.75, 0.77, 0.77, 0.81],
    'F1-Score (Class 1)': [0.68, 0.68, 0.66, 0.68, 0.68, 0.68],
    'Accuracy': [0.89, 0.89, 0.88, 0.88, 0.88, 0.88],
    'AUC': ['-', 0.93, '-', '-', '-', 0.92]
})

print("Models Performance:")
print(results.to_string(index=False))